# Plotted Reports — Complete Inventory
## Energy Market Simulation — Scenario Analysis Framework

This notebook provides a **complete reference** for every visual report generated by the simulation pipeline. Each section contains:

- 📋 **Purpose** — what the chart is trying to communicate
- 📐 **Data sources** — which pipeline objects / DataFrames are read
- 🔣 **Formula** — where KPIs are derived mathematically
- 💻 **Reproducible code** — ready-to-run plot with the live pipeline
- 💡 **Interpretation guidance** — how to read the output

---

### Scenario Inventory

| Scenario | Description | REC | Battery |
|---|---|---|---|
| **A1** | Single supplier, no REC | ✗ | ✗ |
| **A2** | Single supplier + REC | ✓ | ✗ |
| **B1** | Multiple suppliers, no REC | ✗ | ✗ |
| **B2** | Multiple suppliers + REC (with forecasts) | ✓ | ✗ |
| **C3** | Single supplier + REC + Battery optimisation | ✓ | ✓ |

`mixed` variants refer to scenarios using mixed-type participant portfolios (prosumers + consumers together).

---
## Section 1 — Setup & Imports

All downstream cells depend on this section. Run it once before proceeding to any report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings("ignore")

# ── optional: Plotly for Sankey (Report 11) ──
try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠  plotly not installed – Report 11 (Sankey) will be skipped. Run: pip install plotly")

# ── optional: pandapower / networkx for network topology (Report 5) ──
try:
    import pandapower as pp
    import pandapower.plotting as ppplot
    import networkx as nx
    PP_AVAILABLE = True
except ImportError:
    PP_AVAILABLE = False
    print("⚠  pandapower not installed – Report 5 (Network Topology) will be skipped.")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})
EUR_FMT = mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}")
MWH_FMT = mticker.FuncFormatter(lambda v, _: f"{v:,.2f} MWh")

print("✓ Libraries loaded")

In [ ]:
from energy_market_operations import EnergyMarketOperations

# ── Base path of scenario JSON files ──
SCENARIO_DIR = "."
DATA_ROOT    = "../data"      # adjust if your CSV data lives elsewhere

# ── Load and run all relevant scenario pipelines ──
SCENARIOS = {
    "A1": "A1_single_supplier_no_rec.json",
    "A1_mixed": "A1_single_supplier_no_rec_mixed.json",
    "A2": "A2_single_supplier_with_rec.json",
    "A2_mixed": "A2_single_supplier_with_rec_mixed.json",
    "B1": "B1_multiple_supplier_no_rec.json",
    "B1_mixed": "B1_multiple_supplier_no_rec_mixed.json",
    "B2": "B2_multiple_supplier_with_rec.json",
    "B2_mixed": "B2_multiple_supplier_with_rec_mixed.json",
    "B2_forecasts": "B2_multiple_supplier_with_rec_forecasts.json",
    "C3": "C3_single_supplier_rec_battery.json",
}

pipes = {}
for name, json_file in SCENARIOS.items():
    json_path = os.path.join(SCENARIO_DIR, json_file)
    if not os.path.exists(json_path):
        print(f"  ⚠ Not found, skipping: {json_file}")
        continue
    try:
        p = EnergyMarketOperations(json_path, scenario_name=name)
        p.run_all()
        pipes[name] = p
        print(f"  ✓ {name} loaded")
    except Exception as e:
        print(f"  ✗ {name} failed: {e}")

print(f"\n✓ Loaded scenarios: {list(pipes.keys())}")

---
## Report 1 — Supplier Financial Overview (`pipe.plot_financials()`)

**Scope:** All scenario notebooks — A1, A2, B1, B2, C3 (standard + mixed)  
**Source:** `energy_market_operations.py` → `plot_financials()`  
**Chart type:** Two-column: Line chart (left) + Grouped bar chart (right), one row per supplier

### Purpose
Communicate the **profitability composition** of each supplier across simulation periods. The chart enables identification of which revenue streams dominate and how costs evolve month-on-month. Comparing the same panel across A1 vs A2 directly reveals the **financial impact of REC participation** on the supplier.

### What Each Panel Shows
| Panel | Chart | Series |
|---|---|---|
| Left — Financial Overview | Line | Revenue (green), Costs (red), Profit/Loss (blue) in € |
| Right — Revenue Components | Grouped bar | Energy Market Sales, Balancing Rewards, Retail Sales in € |

### Data Source
```
pipe.es_monthly_analysis_df
  → total_revenue_eur, total_costs_eur, profit_loss_eur
  → revenue_energy_market_sales_eur, revenue_balancing_rewards_eur, revenue_retail_sales_eur
```

### Interpretation
- A **widening gap** between Revenue and Costs lines signals improving margin.
- A **dominant Retail Sales bar** indicates strong customer base; a dominant Market Sales bar implies heavy wholesale reliance.
- Under REC scenarios (A2, B2, C3), Retail Sales may **decrease** as internal sharing replaces grid purchases, reducing the supplier's retail billing volume.

In [ ]:
# ── Report 1: Supplier Financial Overview ──────────────────────────────────────
# Select which scenarios to plot; remove any not loaded
report1_scenarios = [s for s in ["A1", "A2", "B1", "B2", "C3"] if s in pipes]

for name in report1_scenarios:
    pipe = pipes[name]
    pipe.plot_financials()

---
## Report 2 — Supplier Balancing Group Position & Costs (`pipe.plot_imbalances()`)

**Scope:** All scenario notebooks — A1, A2, B1, B2, C3 (standard + mixed)  
**Source:** `energy_market_operations.py` → `plot_imbalances()`  
**Chart type:** Two-column: Line chart (left) + Grouped bar chart (right), one row per supplier

### Purpose
Quantify the **quality of market position management**: how well the supplier forecasts net consumption and minimises costly deviations in the balancing market. This reflects operational efficiency of the Day-Ahead and Intraday forecasting stages.

### What Each Panel Shows
| Panel | Chart | Series |
|---|---|---|
| Left — BG Position & Imbalance | Line | BG Actual (red), BG Forecast (green), Imbalance (blue) in MWh |
| Right — Cost Components | Grouped bar | Market Purchases (red), Balancing Penalties (purple), Retail Purchases (orange) in € |

### Data Source
```
pipe.es_monthly_analysis_df
  → balancing_group_actual_mwh, balancing_group_forecast_mwh, imbalance_mwh
  → cost_energy_market_purchases_eur, cost_balancing_penalties_eur, cost_retail_purchases_eur
```

### Interpretation
- **Small imbalance magnitudes** → accurate forecasting, low penalty exposure.
- **Persistent positive imbalance** (Actual > Forecast) → supplier is consistently SHORT; excess purchases in the balancing market at premium prices.
- **High Balancing Penalties bar** → forecasting model needs improvement, or REC-induced uncertainty is degrading BG position.
- In REC scenarios (A2, B2, C3), the corrected net load feeding into the BG reduces the overall position and can **shrink penalties**.

In [ ]:
# ── Report 2: Supplier Balancing Group Position & Costs ────────────────────────
report2_scenarios = [s for s in ["A1", "A2", "B1", "B2", "C3"] if s in pipes]

for name in report2_scenarios:
    pipe = pipes[name]
    pipe.plot_imbalances()

---
## Report 3 — Battery State-of-Charge & Charge/Discharge Profile (C3 only)

**Scope:** C3 scenario only  
**Source:** `C3_single_supplier_rec_battery.ipynb`  
**Chart type:** Two-panel stacked figure (line top, bar bottom) per battery unit

### Purpose
Visualise how the battery storage asset is **dispatched over time** and whether the optimiser respects physical constraints (capacity limits, charge/discharge rate limits). This is the primary diagnostic for battery utilisation quality in the C3 scenario.

### What Each Panel Shows
| Panel | Chart | Series / Reference Lines |
|---|---|---|
| Top — State of Charge | Line | SOC (kWh, blue); dashed red = SOC max; dashed orange = SOC min; dotted grey = initial SOC |
| Bottom — Charge / Discharge | Bar | Charge power (green bars, kW); Discharge power (red bars, kW, negated); dashed lines = max power limits |

**Resolution:** 15-minute intervals across the full simulation horizon.

### Data Source
```
pipe.battery_schedules[battery_id]  (indexed by datetime at 15-min resolution)
  → soc_kwh, charge_kw, discharge_kw
pipe.config['batteries'][i]
  → capacity_kwh, max_charge_kw, max_discharge_kw, min_soc_pct, max_soc_pct, initial_soc_pct
```

### Interpretation
- **Flat SOC line**: battery not being used — check if MILP solver found binding constraints.
- **SOC hitting max repeatedly**: charge-biased optimisation, possibly excess PV generation during midday.
- **SOC hitting min repeatedly**: discharge-biased — suggests high community load or aggressive demand response.
- **Charge and Discharge bars in the same period**: indicates a numerical artefact (round-trip efficiency loss wasting energy) — verify MILP constraint enforcement.
- Battery operation directly determines the `corrected_net_load` seen by the balancing group and therefore impacts imbalance costs (Report 2).

In [ ]:
# ── Report 3: Battery SOC & Charge/Discharge Profile (C3 only) ─────────────────
def plot_battery_profile(pipe, battery_id=None):
    """
    Plot SOC and charge/discharge profile for a given battery.
    Falls back to pipe.battery_schedule_df if battery_schedules dict is not available.
    """
    if pipe.battery_schedules:
        schedules = pipe.battery_schedules
    elif not pipe.battery_schedule_df.empty:
        schedules = {"battery_01": pipe.battery_schedule_df}
    else:
        print("  ⚠ No battery schedule data found.")
        return

    # Fetch battery config specs
    batt_configs = {}
    for batt in pipe.config.get("batteries", pipe.config.get("battery_storage", [])):
        bid = batt.get("battery_id", batt.get("id", "battery_01"))
        batt_configs[bid] = batt

    for bid, df in schedules.items():
        if battery_id and bid != battery_id:
            continue

        batt = batt_configs.get(bid, {})
        cap_kwh     = batt.get("capacity_kwh", batt.get("capacity", 0))
        max_soc_pct = batt.get("max_soc_pct", batt.get("max_soc", 1.0))
        min_soc_pct = batt.get("min_soc_pct", batt.get("min_soc", 0.0))
        init_soc_pct= batt.get("initial_soc_pct", batt.get("initial_soc", 0.5))
        max_chg_kw  = batt.get("max_charge_kw", batt.get("charge_power_kw", 0))
        max_dis_kw  = batt.get("max_discharge_kw", batt.get("discharge_power_kw", 0))
        batt_name   = batt.get("name", bid)

        soc_max_kwh     = cap_kwh * max_soc_pct
        soc_min_kwh     = cap_kwh * min_soc_pct
        initial_soc_kwh = cap_kwh * init_soc_pct

        if not isinstance(df.index, pd.DatetimeIndex):
            df = df.set_index("datetime")
        dates = df.index

        fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
        fig.suptitle(f"{pipe.scenario_name}  |  Battery: {bid} — {batt_name}",
                     fontsize=12, fontweight="bold")

        # ── Top: SOC ──
        axes[0].plot(dates, df["soc_kwh"], color="steelblue", lw=0.9, label="SOC")
        axes[0].axhline(soc_max_kwh, color="red",    ls="--", lw=1.2,
                        label=f"SOC max ({soc_max_kwh:.0f} kWh)")
        axes[0].axhline(soc_min_kwh, color="darkorange", ls="--", lw=1.2,
                        label=f"SOC min ({soc_min_kwh:.0f} kWh)")
        axes[0].axhline(initial_soc_kwh, color="grey", ls=":", lw=1.0,
                        label=f"Initial SOC ({initial_soc_kwh:.0f} kWh)")
        axes[0].set_ylabel("SOC (kWh)")
        axes[0].set_title(f"State of Charge  [Capacity: {cap_kwh} kWh]")
        axes[0].legend(loc="upper right", fontsize=8)
        axes[0].grid(True, alpha=0.3)
        axes[0].fill_between(dates, df["soc_kwh"], soc_min_kwh,
                             where=df["soc_kwh"] >= soc_min_kwh,
                             alpha=0.08, color="steelblue")

        # ── Bottom: Charge / Discharge ──
        width = pd.Timedelta("15min")
        axes[1].bar(dates, df["charge_kw"],    width=width, color="green", alpha=0.75, label="Charge")
        axes[1].bar(dates, -df["discharge_kw"], width=width, color="crimson", alpha=0.75, label="Discharge")
        axes[1].axhline( max_chg_kw, color="green",  ls="--", lw=1.2,
                         label=f"Max charge ({max_chg_kw} kW)")
        axes[1].axhline(-max_dis_kw, color="crimson", ls="--", lw=1.2,
                         label=f"Max discharge ({max_dis_kw} kW)")
        axes[1].axhline(0, color="black", lw=0.6)
        axes[1].set_ylabel("Power (kW)")
        axes[1].set_xlabel("Date")
        axes[1].set_title("Charge / Discharge  [positive = charging, negative = discharging]")
        axes[1].legend(loc="upper right", fontsize=8)
        axes[1].grid(True, alpha=0.3)

        axes[1].xaxis.set_major_locator(mdates.MonthLocator())
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        fig.autofmt_xdate(rotation=45)
        plt.tight_layout()
        plt.show()

if "C3" in pipes:
    plot_battery_profile(pipes["C3"])
else:
    print("C3 scenario not loaded.")

---
## Report 4 — Actual vs. Forecast Load Comparison (Full Year — Aggregated)

**Scope:** Data preparation notebook (`data/loads_res.ipynb`)  
**Source:** Raw CSV profiles (actual, day-ahead forecast, intraday forecast)  
**Chart type:** Two-panel aggregated view — daily aggregated line chart (top) + monthly error bar chart (bottom)

### Purpose
Assess **forecast quality across the entire simulation year** for the full community load. Systematic deviations between actual and forecast loads drive balancing-group imbalances and directly generate the penalty costs visible in Report 2. A full-year view reveals seasonal patterns in forecast error magnitude.

### What the Plot Shows
| Panel | Chart | Series |
|---|---|---|
| Top — Daily Aggregated | Line | Daily total Actual Load, DA Forecast, and Intraday Forecast (MWh/day) |
| Bottom — Monthly MAE | Grouped bar | Mean Absolute Error per month for DA and ID forecasts (MWh) |

### Data Source
```
pipe.es_data["load_forecast_da"]   → day-ahead forecasts per member column
pipe.es_data["load_forecast_id"]   → intraday forecasts per member column
pipe.es_data["load_actual"]        → metered actuals per member column
```

### Interpretation
- **Winter months (Jan, Feb, Nov, Dec)**: typically the highest DA forecast errors due to heating load volatility.
- **Summer months (Jun–Aug)**: lower errors; load profile more predictable.
- **ID MAE consistently below DA MAE**: confirms the intraday trading session adds value by correcting overnight forecast errors.
- A **narrow gap between DA and ID bars** in any month means the intraday market provides little additional information — may signal structural limitations in the forecasting model.


In [ ]:
# ── Report 4: Actual vs. Forecast Load Comparison — Full Year, Aggregated ──────
def plot_forecast_vs_actual_annual(pipe, scenario_name="A1"):
    """
    Two-panel annual aggregated forecast quality report.
    Top:    Daily total community load — Actual vs. DA vs. ID forecasts (MWh/day)
    Bottom: Monthly Mean Absolute Error (MAE) for DA and ID forecasts (MWh)
    """
    ed = pipe.es_data

    # ── Aggregate all member columns to community total ──
    actual  = ed["load_actual"].sum(axis=1)
    da_fcst = ed["load_forecast_da"].sum(axis=1)
    id_fcst = ed["load_forecast_id"].sum(axis=1)

    # ── Daily aggregation ──
    actual_daily  = actual.resample("D").sum()
    da_daily      = da_fcst.resample("D").sum()
    id_daily      = id_fcst.resample("D").sum()

    # ── Monthly MAE ──
    da_error = (actual - da_fcst).abs()
    id_error = (actual - id_fcst).abs()
    da_mae_monthly = da_error.resample("ME").mean()
    id_mae_monthly = id_error.resample("ME").mean()
    month_labels   = da_mae_monthly.index.strftime("%b %Y")

    # ── Annual summary statistics ──
    da_mae_annual = float(da_error.mean())
    id_mae_annual = float(id_error.mean())
    id_improvement = (1 - id_mae_annual / da_mae_annual) * 100 if da_mae_annual > 0 else 0.0

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10))
    fig.suptitle(
        f"{scenario_name}  |  Community Load Forecast Quality — Full Year\n"
        f"Annual DA MAE: {da_mae_annual:.4f} MWh  |  Annual ID MAE: {id_mae_annual:.4f} MWh  |  "
        f"ID Improvement: {id_improvement:.1f}%",
        fontsize=12, fontweight="bold",
    )

    # ── Top: daily aggregated lines ──
    ax1.plot(actual_daily.index,  actual_daily,  color="green",     lw=1.6, alpha=0.9, label="Actual Load")
    ax1.plot(da_daily.index,      da_daily,      color="steelblue", lw=1.4, alpha=0.8, ls="--", label="Day-Ahead Forecast")
    ax1.plot(id_daily.index,      id_daily,      color="orangered", lw=1.4, alpha=0.8, ls=":",  label="Intraday Forecast")
    ax1.fill_between(actual_daily.index,
                     actual_daily, da_daily,
                     where=(actual_daily >= da_daily),
                     interpolate=True, alpha=0.10, color="red", label="DA over-forecast")
    ax1.fill_between(actual_daily.index,
                     actual_daily, da_daily,
                     where=(actual_daily < da_daily),
                     interpolate=True, alpha=0.10, color="blue", label="DA under-forecast")
    ax1.set_ylabel("Daily Total Load (MWh/day)", fontsize=10)
    ax1.set_title("Daily Aggregated: Actual vs. Day-Ahead vs. Intraday Forecast", fontsize=10)
    ax1.legend(ncol=3, fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_locator(mdates.MonthLocator())
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax1.yaxis.set_major_formatter(MWH_FMT)

    # ── Bottom: monthly MAE grouped bars ──
    x = np.arange(len(month_labels))
    bw = 0.38
    bars1 = ax2.bar(x - bw / 2, da_mae_monthly.values, width=bw,
                    color="steelblue", alpha=0.85, label="DA Forecast MAE (MWh)")
    bars2 = ax2.bar(x + bw / 2, id_mae_monthly.values, width=bw,
                    color="orangered", alpha=0.85, label="ID Forecast MAE (MWh)")
    ax2.axhline(da_mae_annual, color="steelblue", ls="--", lw=1.2, alpha=0.7,
                label=f"Annual DA avg ({da_mae_annual:.4f} MWh)")
    ax2.axhline(id_mae_annual, color="orangered", ls="--", lw=1.2, alpha=0.7,
                label=f"Annual ID avg ({id_mae_annual:.4f} MWh)")
    ax2.set_xticks(x)
    ax2.set_xticklabels(month_labels, rotation=35, ha="right", fontsize=8)
    ax2.set_ylabel("Mean Absolute Error (MWh)", fontsize=10)
    ax2.set_title("Monthly DA & ID Forecast MAE", fontsize=10)
    ax2.legend(ncol=2, fontsize=8)
    ax2.grid(axis="y", alpha=0.3)

    # Annotate improvement % per month
    for b1, b2 in zip(bars1, bars2):
        da_v = b1.get_height()
        id_v = b2.get_height()
        if da_v > 0:
            imp = (1 - id_v / da_v) * 100
            ax2.text(
                b2.get_x() + b2.get_width() / 2,
                max(da_v, id_v) * 1.04,
                f"{imp:+.0f}%",
                ha="center", va="bottom", fontsize=6.5,
                color="#27ae60" if imp > 0 else "#e74c3c",
            )

    fig.autofmt_xdate(rotation=35)
    plt.tight_layout()
    plt.show()

    print(f"\n  Annual Summary — {scenario_name}")
    print(f"  {'DA Forecast MAE':<25}: {da_mae_annual:.4f} MWh")
    print(f"  {'ID Forecast MAE':<25}: {id_mae_annual:.4f} MWh")
    print(f"  {'ID Improvement':<25}: {id_improvement:.1f}%")
    print(f"  {'Peak DA Error Month':<25}: {da_mae_monthly.idxmax().strftime('%b %Y')}  ({da_mae_monthly.max():.4f} MWh)")
    print(f"  {'Best DA Error Month':<25}: {da_mae_monthly.idxmin().strftime('%b %Y')}  ({da_mae_monthly.min():.4f} MWh)")

# ── Run for all loaded scenarios ──
for name in list(pipes.keys()):
    plot_forecast_vs_actual_annual(pipes[name], scenario_name=name)


---
## Report 6 — REC Self-Sufficiency Rate (SSR) & Self-Consumption Rate (SCR)

**Scope:** REC-enabled scenarios only — A2, B2, C3  
**Chart type:** Grouped horizontal bar chart (SSR vs SCR per scenario)

### Purpose
Quantify **local matching quality** between REC generation and demand. SSR and SCR are the canonical key performance indicators for energy community effectiveness and are required outputs under Austrian REC regulatory reporting frameworks.

### Mathematical Definition

$$\text{SSR}_r = \frac{\sum_t E^{\text{shared}}_{r,t}}{\sum_t L^{\text{REC}}_{r,t}} \qquad (2.28)$$

$$\text{SCR}_r = \frac{\sum_t E^{\text{shared}}_{r,t}}{\sum_t G^{\text{REC}}_{r,t}} \qquad (2.29)$$

Where:
- $E^{\text{shared}}_{r,t}$ = `internal_shared_energy_mwh` — energy shared within REC at time $t$
- $L^{\text{REC}}_{r,t}$ = total REC load consumption — sum of `actual_load_mwh` for all members
- $G^{\text{REC}}_{r,t}$ = total REC generation — sum of `actual_gen_mwh` for all prosumers

### Data Source
```
pipe.es_timeseries_df  → internal_shared_energy_mwh (per BG per timestamp)
pipe.customer_billing_df  → actual_load_mwh, actual_gen_mwh (per customer per timestamp)
```

### Interpretation
- **SSR close to 1.0 (100%)**: the community is nearly fully self-sufficient — minimal dependence on grid imports.
- **SCR close to 1.0 (100%)**: nearly all generated energy is consumed internally — minimal curtailment/export.
- The **SSR-SCR gap** reflects a generation/load imbalance: high SCR + low SSR → large load but insufficient generation; low SCR + high SSR → large generation surplus.
- Battery optimisation (C3) should **increase both** SSR and SCR relative to C3-no-battery by time-shifting generation to match load peaks.

In [ ]:
# ── Report 6: REC SSR & SCR ─────────────────────────────────────────────────────
def compute_ssr_scr(pipe):
    """Return (SSR, SCR) scalar values for a REC-enabled pipeline."""
    ts = pipe.es_timeseries_df
    billing = pipe.customer_billing_df

    # Shared energy (REC settlement step output)
    shared_mwh = ts["internal_shared_energy_mwh"].sum() if "internal_shared_energy_mwh" in ts.columns else 0.0

    # Total REC load and generation from billing
    total_load_mwh = billing["actual_load_mwh"].sum()
    total_gen_mwh  = billing["actual_gen_mwh"].sum()

    ssr = shared_mwh / total_load_mwh if total_load_mwh > 0 else 0.0
    scr = shared_mwh / total_gen_mwh  if total_gen_mwh  > 0 else 0.0
    return ssr, scr

rec_scenarios = [s for s in ["A2", "A2_mixed", "B2", "B2_mixed", "B2_forecasts", "C3"] if s in pipes]

ssr_vals, scr_vals, labels = [], [], []
for name in rec_scenarios:
    ssr, scr = compute_ssr_scr(pipes[name])
    ssr_vals.append(ssr * 100)
    scr_vals.append(scr * 100)
    labels.append(name)
    print(f"  {name:20s}  SSR={ssr*100:.1f}%   SCR={scr*100:.1f}%")

if labels:
    x = np.arange(len(labels))
    bw = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - bw/2, ssr_vals, width=bw, color="#2980b9", alpha=0.85, label="SSR — Self-Sufficiency Rate")
    bars2 = ax.bar(x + bw/2, scr_vals, width=bw, color="#27ae60", alpha=0.85, label="SCR — Self-Consumption Rate")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 110)
    ax.axhline(100, color="grey", ls="--", lw=0.8, alpha=0.5)
    ax.set_ylabel("Rate (%)")
    ax.set_title("REC Self-Sufficiency Rate (SSR) & Self-Consumption Rate (SCR)\nby Scenario",
                 fontweight="bold")
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.grid(axis="y", alpha=0.3)
    # Annotate bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No REC scenarios loaded.")

---
## Report 7 — REC Member-Level Cost Savings vs. Baseline

**Scope:** REC-enabled scenarios — A2, B2, C3 (compared against their A1/B1 baseline counterparts)  
**Chart type:** Horizontal ranked bar chart with baseline cost scatter overlay

### Purpose
Identify **which individual community members capture the most economic benefit** from REC participation, and whether the distribution of savings is equitable across prosumers and consumers. This directly answers the question: *"Is the REC cost-effective for each participant type?"*

### Mathematical Definition

For each REC participant $i$:

$$\Delta C_i = C^{\text{baseline}}_i - C^{\text{REC}}_i \qquad (2.30)$$

$$\Delta C^{\text{REC}} = \sum_{i=1}^{N_r} \Delta C_i \qquad (2.31)$$

Where:
- $C^{\text{baseline}}_i$ = annual cost from **uncorrected** metered values (retail tariff × original load)
- $C^{\text{REC}}_i$ = annual cost after **REC sharing corrections** (retail tariff × corrected load)

### Data Source
```
pipe.customer_billing_df  → sales_revenue_eur per customer (post-REC)
Baseline: recompute from pipe.es_data["load_actual"] × retail_price
```

### Interpretation
- **Positive $\Delta C_i$**: member saves money — REC reduces their grid import cost.
- **Negative $\Delta C_i$**: unusual; member pays more — possible only if feed-in corrections disadvantage them vs their own generation income.
- **Prosumers typically show larger savings** since their load is offset by shared PV generation from neighbours.
- **Consumers with high base load** benefit most in absolute terms; the per-kWh saving is similar across members but scales with consumption volume.

In [ ]:
# ── Report 7: REC Member-Level Cost Savings vs. Baseline ───────────────────────
def compute_member_cost_savings(pipe_rec, pipe_baseline):
    """
    Compute per-member annual cost saving:
        ΔC_i = C_baseline_i - C_REC_i
    where costs are the total retail charges billed (sales_revenue_eur from supplier side
    = what each customer pays).
    """
    cfg = pipe_rec.config
    prices_rec  = pipe_rec.es_data["prices"]
    prices_base = pipe_baseline.es_data["prices"]

    records = []
    all_members = cfg.get("prosumers", []) + cfg.get("consumers", [])

    for member in all_members:
        cid   = member["meter_id"]
        ctype = "prosumer" if member in cfg.get("prosumers", []) else "consumer"
        sid   = member["supplier"]["supplier_id"]

        # ── Retail price for this supplier ──
        retail_col_rec  = next(s["retail_pricing"]["price"] for s in cfg["suppliers"] if s["supplier_id"] == sid)
        retail_price_rec  = prices_rec[retail_col_rec]
        retail_price_base = prices_base[retail_col_rec] if retail_col_rec in prices_base.columns else retail_price_rec

        # ── Baseline cost: original load × retail price ──
        load_id = member["load"]["id"] if "load" in member and member["load"] else None
        if load_id is None:
            continue

        orig_load    = pipe_baseline.es_data["load_actual"][load_id] if load_id in pipe_baseline.es_data["load_actual"].columns \
                       else pd.Series(0.0, index=retail_price_base.index)
        corr_load    = pipe_rec.corrected_load_df[load_id] if load_id in pipe_rec.corrected_load_df.columns \
                       else orig_load.copy()

        cost_base = (orig_load * retail_price_base).sum()
        cost_rec  = (corr_load * retail_price_rec).sum()

        # ── For prosumers: subtract their feed-in income ──
        if "res" in member and member["res"]:
            gen_id = member["res"]["id"]
            feedin_col = next(s["feedin_pricing"]["price"] for s in cfg["suppliers"] if s["supplier_id"] == sid)
            fi_price = prices_rec[feedin_col]
            orig_gen = pipe_baseline.es_data["res_actual"][gen_id] if gen_id in pipe_baseline.es_data["res_actual"].columns \
                       else pd.Series(0.0, index=fi_price.index)
            corr_gen = pipe_rec.corrected_gen_df[gen_id] if gen_id in pipe_rec.corrected_gen_df.columns \
                       else orig_gen.copy()
            # Under baseline: less corrected gen → more export at feed-in rate
            fi_income_base = (orig_gen * fi_price).sum()
            fi_income_rec  = (corr_gen * fi_price).sum()
            # Net cost = retail - feedin_income
            cost_base -= fi_income_base
            cost_rec  -= fi_income_rec

        records.append({
            "customer_id": cid,
            "customer_type": ctype,
            "cost_baseline_eur": cost_base,
            "cost_rec_eur": cost_rec,
            "saving_eur": cost_base - cost_rec,
        })

    return pd.DataFrame(records)

# ── Compute savings for each REC scenario vs its baseline ──
saving_pairs = [
    ("A2",       "A1",   "A2 vs A1 (standard)"),
    ("A2_mixed", "A1_mixed", "A2 vs A1 (mixed)"),
    ("C3",       "A1",   "C3 vs A1 (battery)"),
]

for rec_name, base_name, title in saving_pairs:
    if rec_name not in pipes or base_name not in pipes:
        print(f"  ⚠ Skipping {title} — scenarios not loaded")
        continue

    df_savings = compute_member_cost_savings(pipes[rec_name], pipes[base_name])
    if df_savings.empty:
        print(f"  ⚠ No member data for {title}")
        continue

    df_s = df_savings.sort_values("saving_eur", ascending=True)
    colors = ["#27ae60" if v >= 0 else "#e74c3c" for v in df_s["saving_eur"]]
    type_markers = {"prosumer": "★", "consumer": "●"}

    fig, ax = plt.subplots(figsize=(11, max(4, len(df_s) * 0.55)))
    bars = ax.barh(df_s["customer_id"], df_s["saving_eur"], color=colors, alpha=0.85, height=0.6)
    ax.scatter(df_s["cost_baseline_eur"], df_s["customer_id"],
               color="darkgrey", zorder=5, s=60, marker="D", label="Baseline Cost (€)")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Annual Cost Saving  ΔC_i  (€)")
    ax.set_title(f"Member-Level Cost Savings — {title}\n"
                 f"ΔC_i = C_baseline − C_REC  |  ▶ positive = member saves money",
                 fontweight="bold")
    ax.legend()
    ax.xaxis.set_major_formatter(EUR_FMT)
    ax.grid(axis="x", alpha=0.3)

    # Annotate bars
    for bar, val in zip(bars, df_s["saving_eur"]):
        ax.text(bar.get_width() + abs(bar.get_width()) * 0.02,
                bar.get_y() + bar.get_height() / 2,
                f"€{val:,.0f}", va="center", fontsize=8)

    # Colour legend
    handles = [Patch(color="#27ae60", label="Saving (positive)"),
               Patch(color="#e74c3c", label="Increased cost (negative)")]
    ax.legend(handles=handles + [plt.scatter([], [], color="darkgrey", marker="D", s=60,
                                             label="Baseline cost reference")])
    plt.tight_layout()
    plt.show()

    total_saving = df_savings["saving_eur"].sum()
    print(f"  Total community saving {title}: €{total_saving:,.2f}")

---
## Report 8 — Community-Level Aggregate Cost Savings (ΔCREC)

**Scope:** All REC-enabled scenarios vs their respective baselines  
**Chart type:** Stacked bar chart (decomposed by individual member) across scenarios

### Purpose
This is the **primary economic figure** for cross-scenario comparison. $\Delta C^{\text{REC}}$ directly answers: *"How much is the community as a whole saving by participating in the REC, and does battery optimisation amplify this?"*

### Mathematical Definition

$$\Delta C^{\text{REC}} = \sum_{i=1}^{N_r} \Delta C_i \qquad (2.31)$$

Each member's contribution to the stack shows the **distribution of economic benefit**, enabling identification of parasitic vs. high-value participants.

### Interpretation
- A **higher total bar** for C3 vs A2 confirms battery optimisation increases community savings.
- If one or two members dominate the stack, savings concentration is high — relevant for regulatory fairness assessments.
- Comparing **standard vs. mixed** variants of the same scenario reveals how participant portfolio composition changes the aggregate saving.

In [ ]:
# ── Report 8: Community-Level Aggregate Cost Savings (ΔCREC) ───────────────────
saving_pairs_all = [
    ("A2",       "A1",       "A2 vs A1"),
    ("A2_mixed", "A1_mixed", "A2 vs A1 mixed"),
    ("C3",       "A1",       "C3 vs A1"),
]

all_savings = {}
for rec_name, base_name, title in saving_pairs_all:
    if rec_name not in pipes or base_name not in pipes:
        continue
    df_s = compute_member_cost_savings(pipes[rec_name], pipes[base_name])
    if not df_s.empty:
        all_savings[title] = df_s

if all_savings:
    # ── Stacked bar decomposed by member ──
    all_titles  = list(all_savings.keys())
    all_members = sorted({m for df in all_savings.values() for m in df["customer_id"].unique()})
    cmap = plt.cm.get_cmap("tab10", len(all_members))
    member_colors = {m: cmap(i) for i, m in enumerate(all_members)}

    fig, ax = plt.subplots(figsize=(max(8, len(all_titles)*2), 6))
    bottoms = np.zeros(len(all_titles))

    for member in all_members:
        vals = []
        for title, df in all_savings.items():
            row = df[df["customer_id"] == member]
            vals.append(row["saving_eur"].values[0] if not row.empty else 0.0)
        ax.bar(all_titles, vals, bottom=bottoms,
               color=member_colors[member], alpha=0.85, label=member)
        bottoms += np.array(vals)

    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("Annual Cost Saving  ΔCREC  (€)")
    ax.set_title("Community-Level Aggregate Cost Savings  ΔCREC\n"
                 "Stacked by individual member contribution",
                 fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    ax.yaxis.set_major_formatter(EUR_FMT)
    ax.grid(axis="y", alpha=0.3)

    # Annotate total
    for i, (title, df) in enumerate(all_savings.items()):
        total = df["saving_eur"].sum()
        ax.text(i, total + abs(total)*0.02, f"€{total:,.0f}",
                ha="center", va="bottom", fontweight="bold", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("No REC savings data available.")

---
## Report 10 — Grid Import vs. REC-Internal Energy Share: With & Without Optimization

**Scope:** A2 (REC, no battery) vs. C3 (REC + battery optimisation)  
**Chart type:** Stacked area chart (time-series at 15-min resolution) + secondary axis for battery dispatch

### Purpose
Directly visualise **how battery optimisation shifts the temporal distribution of grid imports and REC-internal sharing**. This is the operational counterpart to the financial metrics in Reports 7–9 and shows *why* C3 achieves higher SSR/SCR.

### What the Plot Shows
| Layer | Description |
|---|---|
| **Blue area** — Grid Import | Residual consumption drawn from the external grid (MWh per 15-min interval) |
| **Green area** — REC Internal Share | Energy consumed from internal REC generation (MWh per 15-min interval) |
| **Orange line** (secondary axis, C3 only) | Battery net dispatch: positive = discharge, negative = charge (kW) |

### Data Source
```
pipe.corrected_load_df         → post-REC corrected per-member load
pipe.es_data["load_actual"]    → original per-member load (= grid import before correction)
pipe.es_timeseries_df["internal_shared_energy_mwh"]  → time-series shared energy
pipe.battery_schedule_df       → soc_kwh, charge_kw, discharge_kw (C3 only)
```

### Interpretation
- **C3 green area larger than A2**: battery is successfully time-shifting PV generation into evening load peaks, increasing REC utilisation.
- **C3 blue area smaller than A2**: direct confirmation that battery reduces grid dependence.
- **Battery dispatch peaks** (orange line) should align with periods where REC generation > load (charging) and load > REC generation (discharging).
- If battery dispatch shows no improvement in grid import, the MILP optimisation may not be binding or the capacity is too small.

In [ ]:
# ── Report 10: Grid Import vs. REC-Internal Energy Share ───────────────────────
def plot_grid_vs_rec_share(pipe, scenario_name, sample_start="2016-01-01", sample_end="2016-01-14"):
    """
    Stacked area chart: grid import (blue) | REC internal share (green).
    Secondary axis: battery dispatch (orange) if available.
    Uses a sample window for readability; set sample_start/end=None for full horizon.
    """
    ts = pipe.es_timeseries_df.copy()
    ts["datetime"] = pd.to_datetime(ts["datetime"])
    ts = ts.set_index("datetime").sort_index()

    # Aggregate over all BGs
    shared_ts  = ts["internal_shared_energy_mwh"].resample("15min").sum() \
                 if "internal_shared_energy_mwh" in ts.columns \
                 else pd.Series(0.0, index=ts.index)

    # Total actual load (MWh per interval across community)
    billing = pipe.customer_billing_df.copy()
    billing["datetime"] = pd.to_datetime(billing["datetime"])
    total_load = billing.groupby("datetime")["actual_load_mwh"].sum()
    total_load = total_load.sort_index()

    # Grid import = load - shared internally
    shared_aligned = shared_ts.reindex(total_load.index, fill_value=0.0)
    grid_import = (total_load - shared_aligned).clip(lower=0)

    # Sample window
    if sample_start:
        total_load    = total_load.loc[sample_start:sample_end]
        shared_aligned = shared_aligned.loc[sample_start:sample_end]
        grid_import    = grid_import.loc[sample_start:sample_end]

    idx = total_load.index

    fig, ax1 = plt.subplots(figsize=(16, 5))
    ax1.stackplot(idx,
                  [grid_import.values, shared_aligned.values],
                  labels=["Grid Import (MWh)", "REC Internal Share (MWh)"],
                  colors=["#3498db", "#2ecc71"], alpha=0.75)
    ax1.set_ylabel("Energy (MWh per 15-min interval)")
    ax1.set_title(f"{scenario_name}  |  Grid Import vs. REC Internal Energy Share\n"
                  f"Sample: {sample_start} → {sample_end}",
                  fontweight="bold")
    ax1.legend(loc="upper left")
    ax1.grid(alpha=0.3)

    # Secondary axis: battery dispatch (if C3)
    if pipe.has_battery and (pipe.battery_schedules or not pipe.battery_schedule_df.empty):
        ax2 = ax1.twinx()
        sched = (list(pipe.battery_schedules.values())[0]
                 if pipe.battery_schedules else pipe.battery_schedule_df)
        if not isinstance(sched.index, pd.DatetimeIndex):
            sched = sched.set_index("datetime")
        net_dispatch = (sched["discharge_kw"] - sched["charge_kw"])
        if sample_start:
            net_dispatch = net_dispatch.loc[sample_start:sample_end]
        ax2.plot(net_dispatch.index, net_dispatch.values,
                 color="darkorange", lw=1.0, alpha=0.85, label="Battery Net Dispatch (kW)")
        ax2.axhline(0, color="darkorange", lw=0.4, ls="--", alpha=0.4)
        ax2.set_ylabel("Battery Net Dispatch (kW)  [+ = discharge]", color="darkorange")
        ax2.tick_params(axis="y", labelcolor="darkorange")
        ax2.legend(loc="upper right")

    ax1.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout()
    plt.show()

    # Summary stats
    total_load_val   = total_load.sum()
    total_shared_val = shared_aligned.sum()
    total_grid_val   = grid_import.sum()
    print(f"  [{scenario_name}]  Total Load: {total_load_val:.1f} MWh  |  "
          f"Grid Import: {total_grid_val:.1f} MWh  |  REC Share: {total_shared_val:.1f} MWh  |  "
          f"SSR (sample): {total_shared_val/total_load_val*100:.1f}%")

# ── Compare A2 (no battery) vs C3 (with battery) ──
for name in ["A2", "C3"]:
    if name in pipes:
        plot_grid_vs_rec_share(pipes[name], scenario_name=name)

---
## Report 9 — Monthly SSR & SCR Evolution (REC Scenarios)

**Scope:** REC-enabled scenarios — A2, A2_mixed, B2, C3  
**Chart type:** Dual-line chart per scenario (monthly resolution) + grouped bar comparison across scenarios

### Purpose
Track how **self-sufficiency and self-consumption evolve month-by-month** across the simulation year. Annual aggregates (Report 5) hide seasonal dynamics: PV-rich summer months produce excess generation (high SCR, lower SSR) while winter months flip the balance. This is the primary report for understanding **seasonal REC performance patterns**.

### Mathematical Definition

Monthly SSR and SCR are computed using the same formulas as the annual KPIs (§2.28–2.29), restricted to each calendar month $m$:

$$\text{SSR}_{r,m} = \frac{\sum_{t \in m} E^{\text{shared}}_{r,t}}{\sum_{t \in m} L^{\text{REC}}_{r,t}}, \qquad \text{SCR}_{r,m} = \frac{\sum_{t \in m} E^{\text{shared}}_{r,t}}{\sum_{t \in m} G^{\text{REC}}_{r,t}}$$

### Data Source
```
pipe.es_timeseries_df  → internal_shared_energy_mwh  (per BG, per timestamp)
pipe.customer_billing_df  → actual_load_mwh, actual_gen_mwh  (per member, per timestamp)
```

### Interpretation
- **Summer peak in SCR**: high PV output → large share of generation consumed internally; SCR may exceed SSR.
- **Winter dip in SCR**: lower PV generation; what little is produced is nearly all consumed internally but total shared volume shrinks.
- **Steady SSR across months**: indicates community load always exceeds generation — sharing is load-limited.
- **C3 above A2 in both metrics**: confirms battery time-shifting is effective throughout the year, not just in specific seasons.
- Months where SSR > SCR indicate generation is insufficient to cover all load — community still depends heavily on the grid.

In [ ]:
# ── Report 9: Monthly SSR & SCR Evolution ─────────────────────────────────────
def compute_monthly_ssr_scr(pipe):
    """
    Return a DataFrame with monthly SSR and SCR for a REC-enabled pipeline.
    Columns: month (period), shared_mwh, load_mwh, gen_mwh, ssr_pct, scr_pct
    """
    ts = pipe.es_timeseries_df.copy()
    billing = pipe.customer_billing_df.copy()

    ts["datetime"] = pd.to_datetime(ts["datetime"])
    billing["datetime"] = pd.to_datetime(billing["datetime"])

    # Monthly shared energy (summed over all BGs)
    if "internal_shared_energy_mwh" not in ts.columns:
        return pd.DataFrame()
    ts = ts.set_index("datetime")
    shared_monthly = ts["internal_shared_energy_mwh"].resample("ME").sum()

    # Monthly load and generation from billing
    billing = billing.set_index("datetime")
    load_monthly = billing["actual_load_mwh"].resample("ME").sum()
    gen_monthly  = billing["actual_gen_mwh"].resample("ME").sum()

    # Align indices
    idx = shared_monthly.index
    load_monthly = load_monthly.reindex(idx, fill_value=0.0)
    gen_monthly  = gen_monthly.reindex(idx, fill_value=0.0)

    ssr = np.where(load_monthly > 0, shared_monthly / load_monthly * 100, 0.0)
    scr = np.where(gen_monthly  > 0, shared_monthly / gen_monthly  * 100, 0.0)

    return pd.DataFrame({
        "month":       idx.strftime("%b %Y"),
        "shared_mwh":  shared_monthly.values,
        "load_mwh":    load_monthly.values,
        "gen_mwh":     gen_monthly.values,
        "ssr_pct":     ssr,
        "scr_pct":     scr,
    })

rec_scenarios = [s for s in ["A2", "A2_mixed", "B2", "C3"] if s in pipes]
monthly_data = {}

for name in rec_scenarios:
    df = compute_monthly_ssr_scr(pipes[name])
    if not df.empty:
        monthly_data[name] = df

# ── Panel 1: per-scenario monthly line charts ──
for name, df in monthly_data.items():
    x = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(x, df["ssr_pct"], "o-", color="#2980b9", lw=2.0, ms=6, label="SSR — Self-Sufficiency Rate")
    ax.plot(x, df["scr_pct"], "s-", color="#27ae60", lw=2.0, ms=6, label="SCR — Self-Consumption Rate")
    ax.fill_between(x, df["ssr_pct"], df["scr_pct"],
                    where=df["ssr_pct"] >= df["scr_pct"],
                    alpha=0.08, color="#2980b9", label="SSR > SCR (load-limited)")
    ax.fill_between(x, df["ssr_pct"], df["scr_pct"],
                    where=df["ssr_pct"] < df["scr_pct"],
                    alpha=0.08, color="#27ae60", label="SCR > SSR (gen-surplus)")
    ax.axhline(df["ssr_pct"].mean(), color="#2980b9", ls="--", lw=1.0, alpha=0.5,
               label=f"Annual avg SSR ({df['ssr_pct'].mean():.1f}%)")
    ax.axhline(df["scr_pct"].mean(), color="#27ae60", ls="--", lw=1.0, alpha=0.5,
               label=f"Annual avg SCR ({df['scr_pct'].mean():.1f}%)")
    ax.set_xticks(x)
    ax.set_xticklabels(df["month"], rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0, min(110, max(df["ssr_pct"].max(), df["scr_pct"].max()) * 1.15))
    ax.set_ylabel("Rate (%)")
    ax.set_title(f"{name}  |  Monthly SSR & SCR Evolution\n"
                 f"Self-Sufficiency Rate vs. Self-Consumption Rate",
                 fontweight="bold")
    ax.legend(ncol=2, fontsize=8)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# ── Panel 2: cross-scenario SSR comparison (grouped bars by month) ──
if len(monthly_data) > 1:
    all_months = list(list(monthly_data.values())[0]["month"])
    x = np.arange(len(all_months))
    n = len(monthly_data)
    bw = 0.8 / n
    scenario_colors = {"A2": "#2980b9", "A2_mixed": "#5dade2",
                       "B2": "#8e44ad", "C3": "#e67e22"}

    fig, (ax_ssr, ax_scr) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    fig.suptitle("Cross-Scenario Monthly Comparison: SSR (top) & SCR (bottom)",
                 fontsize=12, fontweight="bold")

    for i, (name, df) in enumerate(monthly_data.items()):
        offset = (i - (n - 1) / 2) * bw
        color = scenario_colors.get(name, f"C{i}")
        ax_ssr.bar(x + offset, df["ssr_pct"], width=bw, color=color, alpha=0.82, label=name)
        ax_scr.bar(x + offset, df["scr_pct"], width=bw, color=color, alpha=0.82, label=name)

    for ax, label in [(ax_ssr, "SSR — Self-Sufficiency Rate (%)"),
                      (ax_scr, "SCR — Self-Consumption Rate (%)")]:
        ax.set_ylabel(label)
        ax.legend(fontsize=9)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())
        ax.grid(axis="y", alpha=0.3)

    ax_scr.set_xticks(x)
    ax_scr.set_xticklabels(all_months, rotation=35, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()

---
## Summary — Complete Report Inventory

| # | Report Name | Scenarios | Chart Type | Primary KPI / Output |
|---|---|---|---|---|
| **1** | Supplier Financial Overview | All (A1–C3) | Line + Grouped Bar | Profit/Loss, Revenue decomposition |
| **2** | BG Position & Cost Breakdown | All (A1–C3) | Line + Grouped Bar | Imbalance MWh, Penalty € |
| **3** | Battery SOC & Charge/Discharge | C3 only | Line + Bar (15-min) | SOC utilisation, dispatch adherence |
| **4** | Actual vs. Forecast Load (Full Year) | All | Daily line + Monthly MAE bar | DA/ID forecast MAE by month (MWh) |
| **5** | REC SSR & SCR (Annual) | A2, B2, C3 | Grouped bar | SSR (%), SCR (%) |
| **6** | Member-Level Cost Savings | A2/C3 vs A1 | Horizontal ranked bar | ΔC_i (€ per member) |
| **7** | Community Aggregate Savings ΔCREC | A2/C3 vs A1 | Stacked bar | ΔCREC (€) |
| **8** | Grid Import vs. REC Share (with/without battery) | A2 vs C3 | Stacked area + secondary axis | MWh grid dependency |
| **9** | Monthly SSR & SCR Evolution | A2, A2_mixed, B2, C3 | Dual-line per scenario + cross-scenario grouped bar | Seasonal SSR/SCR patterns (%) |
